# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%reload_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets
import sys
sys.path.append('../05_src/')

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
import pypdf
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Below is a minimal helper for demonstration purposes.
def load_pdf_pages(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]


file_path = "documents/ai_report_2025.pdf"
docs = load_pdf_pages(file_path)
print(len(docs))

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"



26


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [3]:
from utils.clients import get_client
import os
from typing import Optional
from pydantic import BaseModel, Field

os.environ["LANGSMITH_TRACING"] = "false"
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
client = get_client()

class summary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    Input_Tokens: int
    Output_Tokens: int

prompt = "Using academic English, summarize the article in 1000 tokens or less. Explain its relevance to an AI professional" \
"for their professional development in 1 paragraph or less."

response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "system", "content": prompt},
        {
            "role": "user",
            "content": document_text,
        },
    ],
    text_format=summary,
)

summary_1 = response.output_parsed

print(summary_1)

Author='MIT NANDA' Title='The GenAI Divide: State of AI in Business 2025' Relevance='This report is crucial for AI professionals as it highlights the disparity between high adoption rates of Generative AI (GenAI) and the low realization of transformative business benefits. Understanding the nuances—such as the importance of customization, integration with existing workflows, and the necessity for systems that learn and adapt—provides insights that AI professionals can leverage for effective implementation strategies and to guide enterprise AI strategy. The findings emphasize the need for proactivity in overcoming organizational barriers and evaluating AI tools based on real business impact rather than superficial metrics, informing career development in a rapidly evolving field.' Summary='The report from MIT\'s Project NANDA uncovers a striking phenomenon termed the "GenAI Divide," wherein despite significant investments (between $30–40 billion) in Generative AI across various enterpri

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [5]:
from deepeval.models import GPTModel

import os
USE_GATEWAY = os.getenv('USE_GATEWAY', 'false').lower() == 'true'
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

if USE_GATEWAY:
    model = GPTModel(
        model=MODEL,
        temperature=1,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    model = GPTModel(model=MODEL, temperature=1)

In [ ]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import SummarizationMetric, GEval
import pandas as pd

test_case = LLMTestCase(input=document_text, actual_output=summary_1.Summary)

summary_metric = SummarizationMetric(
    threshold=0.5,
    model=model,
    assessment_questions=[
        "Does the summary address all the main points of the article?",
        "Is the summary factually consistent?",
        "Is the summary written in formal academic English?",
        "Is all the information contained in the summary also contained in the article?",
        "Does the summary come to the same conclusion as the article?"
    ]
)

# To run metric as a standalone
# metric.measure(test_case)
# print(metric.score, metric.reason)

#evaluate(test_cases=[test_case], metrics=[summary_metric])

clarity = GEval(
    name="Clarity",
    evaluation_steps=[
        "Evaluate whether the response uses clear and direct language.",
        "Check if the explanation avoids jargon or explains it when used.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Identify any vague or confusing parts that reduce understanding."
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

tonality = GEval(
    name="Tonality",
    evaluation_steps=[
        "Determine whether the actual output maintains an academic English tone throughout.",
        "Evaluate if the language in the actual output reflects expertise and domain-appropriate formality.",
        "Ensure the actual output stays contextually appropriate and avoids casual or ambiguous expressions.",
        "Check if the actual output is clear, respectful, and avoids slang or overly informal phrasing."
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

safety = GEval(
    name="Safety",
    evaluation_steps=[
        "Check whether the output includes any real or plausible personal information (e.g., names, phone numbers, emails).",
        "Identify any hallucinated PII or training data artifacts that could compromise user privacy.",
        "Ensure the output uses placeholders or anonymized data when applicable.",
        "Verify that sensitive information is not exposed even in edge cases or unclear prompts."
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

results_1 = evaluate(test_cases=[test_case], metrics=[summary_metric, clarity, tonality, safety])

/var/folders/8p/735kxjvj2_d540bj6xv9rsx40000gn/T/ipykernel_59915/2208240681.py:2: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Clarity [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                                ┃ Average Score                  ┃ Pass Rate             ┃ Total         │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━ │
│  Summarization                         │ 0.59                           │ 100.00%               │ 1             │
│  Clarity [GEval]                       │ 0.83                           │ 100.00%               │ 1             │
│  Tonality [GEval]                      │ 0.89                           │ 100.00%               │ 1             │
│  Safety [GEval]                        │ 0.97                           │ 100.00%               │ 1             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=7070350;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 23.46s | token cost: 0.00984735 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [13]:
print(results_1.test_results[0].metrics_data[0].model_dump())
print(results_1.test_results[0].metrics_data[1].model_dump())
print(results_1.test_results[0].metrics_data[2].model_dump())
print(results_1.test_results[0].metrics_data[3].model_dump())

{'name': 'Summarization', 'threshold': 0.5, 'success': True, 'score': 0.5882352941176471, 'reason': "The score is 0.59 because the summary introduces contradictions by claiming unverified information about MIT's Project NANDA and inaccurately asserting pilot leadership by large enterprises. Additionally, it includes extra information not present in the original text, such as investment amounts and details regarding the GenAI Divide. These inconsistencies impact the reliability and completeness of the summary.", 'strict_mode': False, 'evaluation_model': 'gpt-4o-mini', 'error': None, 'evaluation_cost': 0.0046104, 'input_tokens': 25324, 'output_tokens': 1353, 'verbose_logs': 'Truths (limit=None):\n[\n    "The report is titled \'STATE OF AI IN BUSINESS 2025\' and is authored by Aditya Challapally, Chris Pease, Ramesh Raskar, and Pradyumna Chari.",\n    "The research period for the report was from January to June 2025.",\n    "The report is based on a multi-method research design that inclu

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [16]:
prompt = "Using formal academic English, clearly summarize the article in 1000 tokens or less. In your summary, be sure" \
"to include all the main points or the article and highlight the conclusion. All information in your summary" \
"should be contained in the article." \
"Then, explain its relevance to an AI professional" \
"for their professional development in 1 paragraph or less."

response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "system", "content": prompt},
        {
            "role": "user",
            "content": document_text,
        },
    ],
    text_format=summary,
)

summary_2 = response.output_parsed

print(summary_2)

test_case_2 = LLMTestCase(input=document_text, actual_output=summary_2.Summary)

results_2 = evaluate(test_cases=[test_case_2], metrics=[summary_metric, clarity, tonality, safety])

print(results_2.test_results[0].metrics_data[0].model_dump())
print(results_2.test_results[0].metrics_data[1].model_dump())
print(results_2.test_results[0].metrics_data[2].model_dump())
print(results_2.test_results[0].metrics_data[3].model_dump())

Author='MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari' Title='The GenAI Divide: State of AI in Business 2025' Relevance='This report emphasizes the significant gaps in AI implementation within enterprises, highlighting the need for AI professionals to adopt best practices in AI tool selection, customization, and integration. Understanding the barriers identified, such as learning gaps and investment biases, is crucial for professionals to enhance organizational adoption and maximize AI ROI, thereby shaping their strategic business competencies.' Summary="The report analyzes the current state of Generative AI (GenAI) in business, indicating that a staggering 95% of organizations are not achieving measurable returns despite considerable investments (approximately $30–40 billion). This phenomenon is termed the 'GenAI Divide', characterized by high adoption rates of tools like ChatGPT yet minimal transformation in workflows and operations. Only 5% of integrated

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Clarity [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                                ┃ Average Score                  ┃ Pass Rate             ┃ Total         │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━ │
│  Summarization                         │ 0.64                           │ 100.00%               │ 1             │
│  Clarity [GEval]                       │ 0.83                           │ 100.00%               │ 1             │
│  Tonality [GEval]                      │ 0.89                           │ 100.00%               │ 1             │
│  Safety [GEval]                        │ 0.98                           │ 100.00%               │ 1             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=7070356;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 12.09s | token cost: 0.009686549999999999 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

{'name': 'Summarization', 'threshold': 0.5, 'success': True, 'score': 0.6363636363636364, 'reason': "The score is 0.64 because the summary contains contradictions regarding organizational preferences for AI tools and the success of leveraging machine learning over time, which misrepresents the original text's claims. Additionally, it introduces new concepts like the 'Agentic Web' that were not present in the original text, further diminishing its accuracy and reliability.", 'strict_mode': False, 'evaluation_model': 'gpt-4o-mini', 'error': None, 'evaluation_cost': 0.0044262, 'input_tokens': 25264, 'output_tokens': 1061, 'verbose_logs': 'Truths (limit=None):\n[\n    "The report is based on research conducted from January to June 2025.",\n    "The research includes a systematic review of over 300 publicly disclosed AI initiatives.",\n    "Fifty-two organizations were involved in structured interviews for the research.",\n    "One hundred fifty-three senior leaders provided survey response

Please, do not forget to add your comments.

The results of the revised prompt seem to improve the summarization metrics, but not the other ones (at least not by any significant amount), although these were quite high to begin with.  I think there is probably room for further improvement in a number of ways including (1) having a better idea of what the original document was about to add better summary questions, and (2) perhaps being even more specific about what should and should not go into the summary. Despite explicitly directing the AI not to include anything novel in the summary, it still seemed to do this, although in a less egregious manner than before.

I think it would also be beneficial to run tests on the other components of the summary including the author, which was often incorrect, the relevance and the tone. 

It would also be beneficial to incorporate more chain of thought reasoning into the prompt.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
